# DeepDFT-Route: Multi-PDK Cross-Technology Training on Google Colab
### Approach 2: PDK Conditioning (45nm Planar + 14nm FinFET)

This notebook trains the Multimodal CNN + GNN fusion model with **PDK Conditioning** so that a single unified neural network simultaneously masters:
1. **45nm Planar CMOS** (UART Transmitter & DFT Multimode Counter across 45%, 55% utilization)
2. **14nm FinFET** (OpenHW CV32E20 / zero-riscy 2-stage microcontroller core)

And evaluates on **completely unseen hold-out circuits**:
- **45nm Test Core:** SPI Master Controller (`spi`)
- **14nm Test Core:** OpenHW CV32E40P (`riscy_cv32e40p` with FPU & 4-stage pipeline)

**PDK Conditioning Highlights:**
- Spatial density maps conditioned via 2D one-hot planes ($12 \to 14$ channels)
- Circuit netlist graphs conditioned via normalized technology node scale ($10 \to 11$ node features)
- Filtered dummy filler/decap cells to prevent density saturation on 14nm FinFET

## 1. Verify GPU & Install Dependencies
Make sure your Colab Runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
# 1. Verify GPU
!nvidia-smi

# 2. Install PyTorch Geometric and required packages
!pip install -q torch-geometric scipy matplotlib
import torch
print(f"PyTorch Version: {torch.__version__}, CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 2. Upload Self-Contained Bundle (`dataset_multi_pdk.tar.gz`)
Upload the compact ~6.9 MB archive `dataset_multi_pdk.tar.gz` located in `DFT/data/`.

In [ ]:
import os
from google.colab import files

if not os.path.exists("dataset_multi_pdk.tar.gz"):
    print("Please upload 'dataset_multi_pdk.tar.gz' (~6.9 MB):")
    uploaded = files.upload()

# Unpack dataset and code bundle
!tar -xzf dataset_multi_pdk.tar.gz

import sys
sys.path.insert(0, os.path.abspath("src"))

print("Bundle extracted successfully!")
!ls -la

## 3. Run Multi-PDK Conditioned Training
Trains the unified network across 45nm + 14nm FinFET layouts for 30 epochs and evaluates out-of-sample on SPI and RISCY.

In [ ]:
import os
os.environ["PYTHONPATH"] = os.path.abspath("src") + ":" + os.environ.get("PYTHONPATH", "")

!python scripts/train_multi_pdk.py \
    --dataset-dir multi_pdk_v001 \
    --output-dir experiments/multi_pdk_conditioned \
    --epochs 30 \
    --batch-size 2 \
    --lr 0.001

## 4. Inspect Evaluation Metrics & Curves

In [ ]:
import json
import matplotlib.pyplot as plt

report_path = "experiments/multi_pdk_conditioned/cross_technology_evaluation.json"
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        results = json.load(f)
    print(json.dumps(results, indent=2))

# Plot training history
hist_path = "experiments/multi_pdk_conditioned/training_history.json"
if os.path.exists(hist_path):
    with open(hist_path, "r") as f:
        history = json.load(f)
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Loss Curves")

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["val_pearson"], label="Val Pearson r", color="green")
    plt.xlabel("Epoch")
    plt.ylabel("Pearson r")
    plt.legend()
    plt.title("Validation Correlation")
    plt.tight_layout()
    plt.show()

## 5. Download Trained Checkpoint (`best_model.pt`)
Download the trained checkpoint back to your local machine (`experiments/fusion/multi_pdk_conditioned_v001/best_model.pt`).

In [ ]:
from google.colab import files
import os

ckpt_path = "experiments/multi_pdk_conditioned/best_model.pt"
if os.path.exists(ckpt_path):
    print(f"Downloading {ckpt_path}...")
    files.download(ckpt_path)
else:
    print("Checkpoint not found. Please verify training finished.")